# Nigeria Healthcare Compliance Quickstart
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comply54/comply54/blob/main/notebooks/02_nigeria_health.ipynb)


This notebook demonstrates how to evaluate healthcare AI-agent actions with `comply54`.

The `NigeriaHealthcareCompliance` sector pack combines regulatory and safety controls covering:

- Nigeria National Health Act 2014
- Nigeria Data Protection Act 2023 for special-category health data
- Medical and Dental Practitioners Act Cap M8 LFN 2004
- FMOH AI in Healthcare Policy Draft 2024
- CBN BVN Framework and NIBSS Rules
- OWASP Agentic AI safety controls

We'll evaluate patient-record access and AI-assisted clinical actions to see how comply54 audits, escalates, or denies actions before execution.

In [ ]:
%pip install comply54

In [ ]:
from comply54.sectors import NigeriaHealthcareCompliance

compliance = NigeriaHealthcareCompliance()

print(compliance.name)
print("Jurisdictions:", compliance.jurisdictions)
print("Regulations:")
for regulation in compliance.regulations:
    print(f"  - {regulation}")

In [ ]:
def show_result(result):
    print(f"Overall decision: {result.overall.upper()}")
    print(f"Blocked: {result.blocked}")
    print(f"Audit ID: {result.audit_id}")
    print(f"Evaluated at: {result.evaluated_at}")

    notable_decisions = [
        decision for decision in result.decisions
        if decision.action != "allow"
    ]

    if notable_decisions:
        print("\nPolicy decisions:")
        for decision in notable_decisions:
            print(f"\n  Pack: {decision.pack}")
            print(f"  Regulation: {decision.regulation}")
            print(f"  Decision: {decision.action.upper()}")

            for message in decision.messages:
                print(f"  Message: {message}")

            if decision.citations:
                print("  Citations:")
                for citation in decision.citations:
                    print(
                        f"    - {citation.document}, "
                        f"{citation.section} ({citation.authority}, {citation.year})"
                    )
    else:
        print("\nNo non-allow policy decisions were triggered.")

## Scenario 1 — Treatment access with documented consent

A healthcare AI assistant accesses a patient's record for treatment purposes after the patient's consent has been documented.

The action can proceed, while comply54 records an `audit` decision for compliance traceability.

In [ ]:
treatment_access = compliance.check(
    action="access_patient_records",
    params={
        "patient_id": "P-12345",
        "record_count": 1,
        "purpose": "treatment",
    },
    context={
        "consent_documented": True,
        "purpose": "treatment",
    },
)

show_result(treatment_access)

## Scenario 2 — Patient-record access without consent

The assistant attempts to access a patient's medical record for treatment without documented patient consent.

Because health records contain sensitive patient information and consent has not been documented, comply54 denies the action.

In [ ]:
no_consent_access = compliance.check(
    action="access_patient_records",
    params={
        "patient_id": "P-12345",
        "record_count": 1,
        "purpose": "treatment",
    },
    context={
        "consent_documented": False,
        "purpose": "treatment",
    },
)

show_result(no_consent_access)

## Scenario 3 — AI diagnosis without clinician oversight

The AI generates a clinical assessment without a human clinician supervising the diagnostic process.

Under the healthcare compliance rules, an AI system cannot independently produce a clinical diagnosis without human clinician oversight, so the action is denied.

In [ ]:
unreviewed_diagnosis = compliance.check(
    action="generate_diagnosis",
    params={
        "patient_id": "P-12345",
        "symptoms": "persistent headache, dizziness",
    },
    context={
        "consent_documented": True,
        "human_clinician_oversight": False,
    },
)

show_result(unreviewed_diagnosis)

## Scenario 4 — AI diagnosis with clinician oversight

The same AI-assisted diagnosis is performed with human clinician oversight enabled.

Instead of being denied outright, the action is escalated because the AI-generated diagnosis still requires review and sign-off by a licensed clinician before it can be communicated to the patient.

In [ ]:
reviewed_diagnosis = compliance.check(
    action="generate_diagnosis",
    params={
        "patient_id": "P-12345",
        "symptoms": "persistent headache, dizziness",
    },
    context={
        "consent_documented": True,
        "human_clinician_oversight": True,
    },
)

show_result(reviewed_diagnosis)

## What happens next — LangGraph integration

Direct compliance checks are useful for evaluating individual healthcare actions, but an AI-agent workflow can enforce these checks automatically before a tool executes.

comply54 provides `Comply54Guard`, which can sit between the agent and healthcare tools in a LangGraph workflow:

```python
from comply54.sectors import NigeriaHealthcareCompliance
from comply54.langchain import Comply54Guard, comply54_route
from langgraph.graph import StateGraph
from langgraph.prebuilt import ToolNode

compliance = NigeriaHealthcareCompliance()

guard = Comply54Guard(
    compliance=compliance,
    context={
        "consent_documented": True,
        "human_clinician_oversight": True,
        "licensed_clinician_approval": False,
        "purpose": "treatment",
    },
)

graph = StateGraph(HealthAgentState)

graph.add_node("agent", call_model)
graph.add_node("comply54_guard", guard)
graph.add_node("tools", ToolNode(health_tools))

graph.add_conditional_edges(
    "comply54_guard",
    comply54_route,
    {"tools": "tools", "agent": "agent"},
)
```

The guard evaluates proposed healthcare tool calls before they reach the tool node. This allows actions such as unauthorized record access or unsupervised AI diagnosis to be denied or escalated before execution.

> **Note:** The executable examples in this notebook require only `comply54`. Running the LangGraph integration itself requires the optional LangGraph dependencies (`comply54[langgraph]`). The snippet above is an integration sketch; `HealthAgentState`, `call_model`, and `health_tools` are application-specific components.